# Trusted Zone — Processing pipeline

Same flow as [`trusted_zone_processing.py`](../trusted_zone_processing.py):

1. **Spark batch (Docker):** landing metadata QA → `metadata/pending_workset.json`
2. **Host Python:** cymatics image + video, peak metadata → trusted-zone bucket
3. Append trusted-zone CSV + Parquet
4. Sync Parquet → Delta Lake

Spectral / MFCC features run later in exploitation zone.

## Paths and environment

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
TRUSTED_ZONE_DIR = _root / "trusted_zone"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if str(TRUSTED_ZONE_DIR) not in sys.path:
    sys.path.insert(1, str(TRUSTED_ZONE_DIR))

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("TRUSTED_ZONE_BUCKET =", os.environ.get("TRUSTED_ZONE_BUCKET", "trusted-zone"))

## Imports

In [ ]:
import importlib
from minio.error import S3Error

import trusted_zone_processing as tz
importlib.reload(tz)

from shared.minio_helpers import (
    MINIO_ACCESS_KEY,
    MINIO_SECRET_KEY,
    MINIO_ENDPOINT,
    create_minio_client,
    is_unreachable_minio,
    append_rows_to_csv,
    update_parquet,
)
from shared.sync_delta import sync_observations_to_delta
from spark_trusted_zone import load_pending_workset, run_spark_trusted_zone_subprocess

## [1/4] MinIO client and trusted bucket

Requires `MINIO_ACCESS_KEY` and `MINIO_SECRET_KEY` in `.env`. Start stack: `docker compose up -d`.

In [ ]:
if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise RuntimeError("Set MINIO_ACCESS_KEY and MINIO_SECRET_KEY")

client = create_minio_client()
try:
    tz.ensure_trusted_bucket(client)
except Exception as e:
    if is_unreachable_minio(e):
        raise RuntimeError(
            f"MinIO at {MINIO_ENDPOINT!r} is unreachable while preparing {tz.TRUSTED_BUCKET!r}."
        ) from e
    if isinstance(e, S3Error):
        code = getattr(e, "code", "") or ""
        raise RuntimeError(
            f"Cannot access trusted bucket {tz.TRUSTED_BUCKET!r} (MinIO code={code})."
        ) from e
    raise

## [2/4] Spark batch — landing metadata → pending workset

In [ ]:
print("\n[1/3] Spark batch (Docker): landing-zone metadata → pending trusted workset...")
try:
    run_spark_trusted_zone_subprocess(project_root=PROJECT_ROOT)
    pending = load_pending_workset(client, bucket=tz.TRUSTED_BUCKET)
except Exception as e:
    raise RuntimeError("Trusted-zone Spark batch failed — " + str(e)) from e

if not pending:
    print(
        "  No pending records — empty landing metadata, invalid rows filtered out,"
        " or all UUIDs already in trusted-zone.",
    )
else:
    print(f"  Pending rows after Spark metadata batch: {len(pending)}")
    by_source = {}
    for r in pending:
        s = r.get("source", "unknown")
        by_source[s] = by_source.get(s, 0) + 1
    print(f"\n[2/3] Will process {len(pending)} new records:")
    for s, cnt in sorted(by_source.items()):
        print(f"    {s}: {cnt}")

## [3/4] Cymatics enrichment per pending row

In [ ]:
rows = []
if pending:
    for i, rec in enumerate(pending):
        uid = rec.get("uuid", "?")[:8]
        src = rec.get("source", "?")
        print(f"\n  [{i+1}/{len(pending)}] uuid={uid}...  source={src}")
        row = tz.process_record(client, rec)
        if row:
            rows.append(row)
            print(
                f"    Done — {row['peak_frequency_hz']:.0f} Hz  "
                f"({row['processing_time(seconds)']:.1f}s)"
            )

if not rows:
    print("\n  No records processed.")
else:
    print(f"\n  Enriched {len(rows)} record(s).")

## [4/4] Write metadata and sync Delta

In [ ]:
if rows:
    print(f"\n[3/3] Writing metadata ({len(rows)} rows)...")
    append_rows_to_csv(
        client,
        tz.TRUSTED_BUCKET,
        rows,
        tz.TRUSTED_METADATA_FIELDS,
        key=tz.TRUSTED_META_KEY,
    )
    print(f"  Updated: {tz.TRUSTED_META_KEY}")
    update_parquet(client, tz.TRUSTED_BUCKET, rows)

    print("\n[4/4] Syncing Parquet → Delta Lake...")
    sync_observations_to_delta(client, tz.TRUSTED_BUCKET, zone_label="trusted")

    print("\n  Trusted-zone processing complete.")
    print(f"   Processed: {len(rows)} records")
    print(f"   Bucket: {tz.TRUSTED_BUCKET}")
    print(f"   Metadata: {tz.TRUSTED_META_KEY}")